# Помесячная сводка расходов на резервы

Ноутбук собирает из широкого файла помесячную сводку в разрезе **Актив / УО**.

Для каждой даты считаются:

- **Расходы на резервы** = Ухудшение качества + Переоценка + Движение портфеля
- **Движение хорошего портфеля**
- **Движение среднего портфеля**
- **Движение плохого портфеля**
- **Изменение финсостояния** = Пришла НИ + Ушла НИ + Пришел ПФН + Ушел ПФН + Изменение рестры
- **Изменение обеспеченности**

Категории движения:

**Плохой портфель**
- ПФН + необеспеченный
- или ГР 5–6
- или рестра

**Средний портфель**
- если не плохой:
- ПФН + любая обеспеченность кроме необеспеченного
- или НИ + необеспеченный

**Хороший портфель**
- все остальное


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Настройки

In [ ]:
INPUT_FILE = Path("reserve_expenses_monthly.xlsx")
OUTPUT_FILE = Path("reserve_expenses_summary.xlsx")

SHEET_NAME = 0

OPERATION_COLUMN_CANDIDATES = [
    "Тип операции",
    "Актив/УО",
    "Актив (УО)",
    "Актив или УО",
]


## 2. Вспомогательные функции

In [ ]:
def normalize_name(value):
    s = str(value).replace("\xa0", " ").replace("\n", " ").replace("\r", " ")
    s = s.strip().lower().replace("ё", "е")
    s = re.sub(r"\s+", " ", s)
    return s


def find_any_column(df, candidates):
    normalized = {normalize_name(col): col for col in df.columns}

    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]

    raise KeyError(
        f"Не найдена колонка. Ожидался один из вариантов: {candidates}\n"
        f"Колонки в файле:\n{list(df.columns)}"
    )


def find_month_column(df, prefixes, date):
    if isinstance(prefixes, str):
        prefixes = [prefixes]

    normalized_columns = {
        normalize_name(col): col
        for col in df.columns
    }

    for prefix in prefixes:
        target = normalize_name(f"{prefix}_{date}")
        if target in normalized_columns:
            return normalized_columns[target]

    return None


def parse_number(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0)

    s = (
        series.astype(str)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.replace("%", "", regex=False)
    )

    return pd.to_numeric(s, errors="coerce").fillna(0)


def parse_flag(series):
    s = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(",", ".", regex=False)
    )

    mapping = {
        "1": 1.0,
        "1.0": 1.0,
        "да": 1.0,
        "yes": 1.0,
        "true": 1.0,
        "0": 0.0,
        "0.0": 0.0,
        "нет": 0.0,
        "no": 0.0,
        "false": 0.0,
        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
    }

    out = s.map(mapping)
    numeric = pd.to_numeric(s, errors="coerce")
    return out.fillna(numeric)


def normalize_operation(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("ё", "е")
    )

    return pd.Series(
        np.select(
            [
                s.str.contains("уо", regex=False),
                s.str.contains("актив", regex=False),
            ],
            [
                "УО",
                "Актив",
            ],
            default=series.astype(str).str.strip(),
        ),
        index=series.index,
    )


def normalize_collateral(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("ё", "е")
    )

    unsecured = (
        s.str.contains(
            r"\bне\s*обеспеч|необеспеч",
            regex=True,
            na=False,
        )
        & ~s.str.contains(
            "недостаточно",
            regex=False,
            na=False,
        )
    )

    result = pd.Series(
        np.where(
            unsecured,
            "Необеспеченный",
            "Остальная обеспеченность",
        ),
        index=series.index,
        dtype="object",
    )

    result[s.eq("")] = np.nan
    return result


def sort_dates(date_strings):
    parsed = pd.to_datetime(
        pd.Series(date_strings),
        dayfirst=True,
        errors="coerce",
    )

    temp = pd.DataFrame({
        "date": date_strings,
        "parsed": parsed,
        "order": range(len(date_strings)),
    })

    temp["_missing"] = temp["parsed"].isna()

    return (
        temp.sort_values(
            ["_missing", "parsed", "order"],
            kind="stable",
        )["date"]
        .tolist()
    )


## 3. Чтение файла и поиск дат

In [ ]:
df = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME,
)

COL_OPERATION = find_any_column(
    df,
    OPERATION_COLUMN_CANDIDATES,
)

dates = []

quality_prefix = normalize_name("Ухудшение качества_")

for col in df.columns:
    col_norm = normalize_name(col)

    if col_norm.startswith(quality_prefix):
        col_str = str(col).replace("\xa0", " ").strip()

        prefix_len = len("Ухудшение качества_")
        date = col_str[prefix_len:].strip()

        if date:
            dates.append(date)

dates = list(dict.fromkeys(dates))
dates = sort_dates(dates)

print(f"Строк: {len(df):,}")
print(f"Колонка типа операции: {COL_OPERATION}")
print(f"Найдено дат: {len(dates)}")

for date in dates:
    print(date)


## 4. Состояние портфеля для классификации движения

Для каждой даты используются:

- `НИ_{дата}`
- `ПФН_{дата}`
- `рестра_{дата}`
- `Обеспеченность_{дата}`
- `ГР_{дата}`

Если на текущую дату все пять полей пустые, берется состояние предыдущей доступной даты.


In [ ]:
state_by_date = {}

previous_state = None

for date in dates:

    col_ni = find_month_column(df, ["НИ"], date)
    col_pfn = find_month_column(df, ["ПФН"], date)
    col_restra = find_month_column(
        df,
        ["рестра", "Рестра", "Реструктуризация"],
        date,
    )
    col_collateral = find_month_column(
        df,
        ["Обеспеченность"],
        date,
    )
    col_gr = find_month_column(
        df,
        ["ГР", "Группа риска"],
        date,
    )

    month_state_cols = {
        "НИ": col_ni,
        "ПФН": col_pfn,
        "рестра": col_restra,
        "Обеспеченность": col_collateral,
        "ГР": col_gr,
    }

    missing = [
        name
        for name, col in month_state_cols.items()
        if col is None
    ]

    if missing:
        raise KeyError(
            f"Для даты {date} не найдены колонки состояния: {missing}"
        )

    state = pd.DataFrame(index=df.index)

    state["НИ"] = parse_flag(df[col_ni])
    state["ПФН"] = parse_flag(df[col_pfn])
    state["Рестра"] = parse_flag(df[col_restra])
    state["Обеспеченность"] = normalize_collateral(
        df[col_collateral]
    )
    state["ГР"] = pd.to_numeric(
        df[col_gr],
        errors="coerce",
    )

    state_missing = (
        state["НИ"].isna()
        & state["ПФН"].isna()
        & state["Рестра"].isna()
        & state["Обеспеченность"].isna()
        & state["ГР"].isna()
    )

    if previous_state is not None:
        for col in state.columns:
            state.loc[state_missing, col] = (
                previous_state.loc[state_missing, col]
            )

    state_by_date[date] = state
    previous_state = state.copy()


## 5. Расчет помесячной сводки

In [ ]:
summary_by_date = {}
detail_by_date = {}

operation = normalize_operation(
    df[COL_OPERATION]
)

for date in dates:

    factor_columns = {
        "Пришла НИ": find_month_column(
            df, ["Пришла НИ"], date
        ),
        "Ушла НИ": find_month_column(
            df, ["Ушла НИ"], date
        ),
        "Пришел ПФН": find_month_column(
            df,
            ["Пришел ПФН", "Пришёл ПФН"],
            date,
        ),
        "Ушел ПФН": find_month_column(
            df,
            ["Ушел ПФН", "Ушёл ПФН"],
            date,
        ),
        "Изменение рестры": find_month_column(
            df,
            ["изменение рестры", "Изменение рестры"],
            date,
        ),
        "Изменение обеспеченности": find_month_column(
            df,
            [
                "изменилась обеспеченность",
                "Изменилась обеспеченность",
                "изменение обеспеченности",
                "Изменение обеспеченности",
            ],
            date,
        ),
        "Ухудшение качества": find_month_column(
            df,
            ["Ухудшение качества"],
            date,
        ),
        "Переоценка": find_month_column(
            df,
            ["Переоценка"],
            date,
        ),
        "Движение портфеля": find_month_column(
            df,
            ["Движение портфеля"],
            date,
        ),
    }

    missing = [
        name
        for name, col in factor_columns.items()
        if col is None
    ]

    if missing:
        raise KeyError(
            f"Для даты {date} не найдены факторные колонки: {missing}"
        )

    tmp = pd.DataFrame(index=df.index)

    tmp["Дата"] = date
    tmp["Тип операции"] = operation

    for name, col in factor_columns.items():
        tmp[name] = parse_number(df[col])

    # --------------------------------------------------------
    # Основные агрегаты
    # --------------------------------------------------------

    tmp["Расходы на резервы"] = (
        tmp["Ухудшение качества"]
        + tmp["Переоценка"]
        + tmp["Движение портфеля"]
    )

    tmp["Изменение финсостояния"] = (
        tmp["Пришла НИ"]
        + tmp["Ушла НИ"]
        + tmp["Пришел ПФН"]
        + tmp["Ушел ПФН"]
        + tmp["Изменение рестры"]
    )

    # --------------------------------------------------------
    # Состояние на дату
    # --------------------------------------------------------

    state = state_by_date[date]

    tmp["НИ"] = state["НИ"]
    tmp["ПФН"] = state["ПФН"]
    tmp["Рестра"] = state["Рестра"]
    tmp["Обеспеченность"] = state["Обеспеченность"]
    tmp["ГР"] = state["ГР"]

    is_unsecured = (
        tmp["Обеспеченность"]
        == "Необеспеченный"
    )

    is_other_collateral = (
        tmp["Обеспеченность"].notna()
        & ~is_unsecured
    )

    # --------------------------------------------------------
    # Плохой портфель
    # --------------------------------------------------------

    is_bad = (
        (
            tmp["ПФН"].eq(1)
            & is_unsecured
        )
        |
        tmp["ГР"].isin([5, 6])
        |
        tmp["Рестра"].eq(1)
    )

    # --------------------------------------------------------
    # Средний портфель
    # --------------------------------------------------------

    is_medium_raw = (
        (
            tmp["ПФН"].eq(1)
            & is_other_collateral
        )
        |
        (
            tmp["НИ"].eq(1)
            & is_unsecured
        )
    )

    is_medium = (
        ~is_bad
        & is_medium_raw
    )

    tmp["Категория портфеля"] = np.select(
        [
            is_bad,
            is_medium,
        ],
        [
            "Плохой",
            "Средний",
        ],
        default="Хороший",
    )

    # --------------------------------------------------------
    # Разбивка движения
    # --------------------------------------------------------

    tmp["Движение хорошего портфеля"] = np.where(
        tmp["Категория портфеля"].eq("Хороший"),
        tmp["Движение портфеля"],
        0.0,
    )

    tmp["Движение среднего портфеля"] = np.where(
        tmp["Категория портфеля"].eq("Средний"),
        tmp["Движение портфеля"],
        0.0,
    )

    tmp["Движение плохого портфеля"] = np.where(
        tmp["Категория портфеля"].eq("Плохой"),
        tmp["Движение портфеля"],
        0.0,
    )

    tmp = tmp[
        tmp["Тип операции"].isin(["Актив", "УО"])
    ].copy()

    tmp["Контроль движения"] = (
        tmp["Движение портфеля"]
        - (
            tmp["Движение хорошего портфеля"]
            + tmp["Движение среднего портфеля"]
            + tmp["Движение плохого портфеля"]
        )
    )

    detail_by_date[date] = tmp

    columns_to_sum = [
        "Расходы на резервы",
        "Движение хорошего портфеля",
        "Движение среднего портфеля",
        "Движение плохого портфеля",
        "Изменение финсостояния",
        "Изменение обеспеченности",
    ]

    summary_date = (
        tmp
        .groupby("Тип операции")[columns_to_sum]
        .sum()
        .reindex(
            ["Актив", "УО"],
            fill_value=0,
        )
        .reset_index()
    )

    summary_date.insert(
        0,
        "Дата",
        date,
    )

    summary_by_date[date] = summary_date


print(
    f"Сформировано дат: {len(summary_by_date)}"
)


## 6. Общая сводная таблица

In [ ]:
summary = pd.concat(
    summary_by_date.values(),
    ignore_index=True,
)

display(summary)


## 7. Контроль

In [ ]:
control_rows = []

for date, tmp in detail_by_date.items():
    control_rows.append({
        "Дата": date,
        "Движение портфеля": tmp["Движение портфеля"].sum(),
        "Движение хорошего": tmp["Движение хорошего портфеля"].sum(),
        "Движение среднего": tmp["Движение среднего портфеля"].sum(),
        "Движение плохого": tmp["Движение плохого портфеля"].sum(),
        "Контроль движения": tmp["Контроль движения"].sum(),
    })

control = pd.DataFrame(control_rows)

display(control)


## 8. Сохранение в Excel

Используется `openpyxl`.

В выходном файле:
- лист **Свод** — все даты и Актив/УО;
- лист **Контроль**;
- отдельный лист для каждой даты.


In [ ]:
with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Свод",
        index=False,
    )

    control.to_excel(
        writer,
        sheet_name="Контроль",
        index=False,
    )

    for date, summary_date in summary_by_date.items():

        sheet_name = str(date)

        for ch in ["/", "\\", ":", "*", "?", "[", "]"]:
            sheet_name = sheet_name.replace(ch, ".")

        sheet_name = sheet_name[:31]

        summary_date.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )

    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"

        for column_cells in ws.columns:
            values = [
                "" if cell.value is None else str(cell.value)
                for cell in column_cells
            ]

            max_len = max(
                [len(v) for v in values],
                default=10,
            )

            width = min(
                max(max_len + 2, 10),
                40,
            )

            ws.column_dimensions[
                column_cells[0].column_letter
            ].width = width


print(f"Готово: {OUTPUT_FILE.resolve()}")
